In [2]:
!wget -O kinematics_2d.json "https://raw.githubusercontent.com/TraizeCPerin/Final-Project-CS2/main/data/kinematics_2d.json"

--2026-03-03 11:24:40--  https://raw.githubusercontent.com/TraizeCPerin/Final-Project-CS2/main/data/kinematics_2d.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1837 (1.8K) [text/plain]
Saving to: ‘kinematics_2d.json’

kinematics_2d.json  100%[===================>]   1.79K  --.-KB/s    in 0s      

2026-03-03 11:24:40 (25.8 MB/s) - ‘kinematics_2d.json’ saved [1837/1837]



In [3]:
# Install firebase library if not installed
!pip install firebase-admin

import json
import math
import firebase_admin
from firebase_admin import credentials
from firebase_admin import db

# Load JSON data
with open("kinematics_2d.json") as f:
    data = json.load(f)

# Firebase setup
cred = credentials.Certificate("firebase_key.json")
firebase_admin.initialize_app(cred, {
    'databaseURL': 'https://rosal-kinematics-db-default-rtdb.asia-southeast1.firebasedatabase.app/'
})
ref = db.reference("/projectiles")  # Root node in your DB

# Physics functions
def max_height(v0, angle_deg, g):
    angle_rad = math.radians(angle_deg)
    return (v0**2 * (math.sin(angle_rad)**2)) / (2 * abs(g))

def horizontal_range(v0, angle_deg, g):
    angle_rad = math.radians(angle_deg)
    return (v0**2 * math.sin(2*angle_rad)) / abs(g)

# Calculate and push to Firebase
results = []

for scenario in data:
    v0 = scenario["initial_conditions"]["v0"]
    angle = scenario["initial_conditions"]["angle_deg"]
    g = scenario["initial_conditions"]["g"]

    h_max = max_height(v0, angle, g)
    x_range = horizontal_range(v0, angle, g)

    scenario_result = {
        "scenario_id": scenario["scenario_id"],
        "max_height": round(h_max, 2),
        "horizontal_range": round(x_range, 2)
    }
    results.append(scenario_result)

    # Push to Firebase
    ref.child(f"scenario_{scenario['scenario_id']}").set(scenario_result)

# Display results in Colab
for r in results:
    print(f"Scenario {r['scenario_id']} → Max Height: {r['max_height']} m, Horizontal Range: {r['horizontal_range']} m")

Scenario 1 → Max Height: 10.19 m, Horizontal Range: 40.77 m
Scenario 2 → Max Height: 2.87 m, Horizontal Range: 19.86 m
Scenario 3 → Max Height: 23.89 m, Horizontal Range: 55.17 m
